# Load new data with polynomial wavelength calibration

This notebook:
1. Loads a FITS cube (193791.fits)
2. Loads the polynomial calibration from bh_avecal.json
3. Gets central wavelength (CW) for the FITS
4. Applies wavelength calibration to the data
5. Plots spectra (same style as 03_load_spectral_data)

In [ ]:
import numpy as np
from astropy.io import fits

from bh_molecule.instruments import (
    Vis133M,
    load_bh_wavecal_json,
    apply_polynomial_wavecal,
    get_cw_from_header,
    estimate_cw_from_features,
)

FITS_PATH = "193791.fits"
WAVECAL_JSON = "bh_avecal.json"

In [ ]:
# a) Load 193791.fits
hdu = fits.open(FITS_PATH)[0]
cube = np.asarray(hdu.data, dtype=float)
header = dict(hdu.header)
assert cube.ndim == 3, f"Expected 3D cube, got {cube.ndim}D"
F, C, P = cube.shape
print(f"Cube shape: {F} frames × {C} channels × {P} pixels")

In [ ]:
# b) Load bh_avecal.json polynomial
wavecal = load_bh_wavecal_json(path=WAVECAL_JSON)
print("Wavecal keys:", list(wavecal.keys()))
print("reference_cw_nm:", wavecal["reference_cw_nm"])

In [ ]:
# c) Get CW for 193791.fits (from header or estimate from spectrum)
cw_nm = get_cw_from_header(header)
if cw_nm is None:
    # Fallback: estimate from brightest pixel using calibration polynomial
    cw_nm = estimate_cw_from_features(cube, wavecal=wavecal)
    print(f"CW estimated from spectrum: {cw_nm:.4f} nm")
else:
    print(f"CW from FITS header: {cw_nm:.4f} nm")

In [ ]:
# d) Apply wavelength calibration to this data
# Get wavelength axis (nm) for this cube's pixel count and CW
wl_nm = apply_polynomial_wavecal(P, cw_nm=cw_nm, wavecal=wavecal)

# Build per-channel slopes/intercepts (same dispersion for all channels) so Vis133M can plot
x = np.arange(P, dtype=float)
coefs = np.polyfit(x, wl_nm, 1)
slopes = np.full(C, coefs[0])
intercepts = np.full(C, coefs[1])

# Vis133M with formula wavecal uses these for wl_nm
s = Vis133M(
    FITS_PATH,
    wavecal=None,
    wavecal_mode="formula",
    slopes=slopes,
    intercepts=intercepts,
)
print("Wavelength calibration applied. Ready to plot.")

In [ ]:
# e) Plot spectra like in 03_load_spectral_data (frame 31, channel 30, dark theme)
fig = s.plot_spectrum_plotly(31, 30, theme='dark')
fig.show()